# Rule-Based Career Decision System with DAG-Based Learning Path
This notebook implements the Rule-Based Decision Engine and the DAG-based learning path generator (cosine similarity is not part of this module). MongoDB and Flask sections are included but disabled by default — toggle the flags below to enable them on the deployment machine.

## 1. Import Libraries and Setup Environment
Import required Python libraries: pandas, numpy, networkx, json.

In [1]:
import os
import re
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, field

import pandas as pd
import numpy as np
import networkx as nx

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Config flags - set to True on the machine where MongoDB / Flask are set up
USE_MONGODB = False
RUN_FLASK_APP = False

DATA_PATH = "jobs_dataset_skills_final.csv"
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB_NAME = "career_recommendation"

print("Config loaded")

Config loaded


## 2. Text Normalization and Skill Standardization
Clean and standardize skill text before matching.

In [3]:
def normalize_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_skill_list(raw: Any) -> List[str]:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    parts = [normalize_text(p) for p in str(raw).split(",")]
    return sorted({p for p in parts if p})

print(normalize_text("  Machine-Learning, TensorFlow!! "))
print(parse_skill_list("Python, SQL, Data Visualization, python"))

machine-learning tensorflow
['data visualization', 'python', 'sql']


## 3. Load and Inspect Job Dataset
Load job data from CSV. Falls back to a small sample dataset if the CSV isn't found.

In [4]:
def load_job_dataset(csv_path: str) -> pd.DataFrame:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"Loaded {len(df)} job postings from '{csv_path}'")
    else:
        print(f"'{csv_path}' not found - using built-in sample dataset")
        df = pd.DataFrame([
            {"job_title": "Backend Developer I", "role_category": "backend_developer",
             "skills_str": "python, sql, docker, aws"},
            {"job_title": "Backend Developer II", "role_category": "backend_developer",
             "skills_str": "python, sql, docker, aws, kubernetes"},
            {"job_title": "Backend Developer III", "role_category": "backend_developer",
             "skills_str": "python, sql, aws"},
            {"job_title": "Data Analyst I", "role_category": "data_analyst",
             "skills_str": "python, sql, excel, statistics, tableau"},
            {"job_title": "Data Analyst II", "role_category": "data_analyst",
             "skills_str": "python, sql, excel, power bi"},
            {"job_title": "Data Analyst III", "role_category": "data_analyst",
             "skills_str": "sql, excel, statistics"},
            {"job_title": "Machine Learning Engineer I", "role_category": "ml_engineer",
             "skills_str": "python, machine learning, tensorflow, scikit-learn, statistics"},
            {"job_title": "Machine Learning Engineer II", "role_category": "ml_engineer",
             "skills_str": "python, machine learning, tensorflow, deep learning"},
            {"job_title": "Machine Learning Engineer III", "role_category": "ml_engineer",
             "skills_str": "python, machine learning, scikit-learn"},
            {"job_title": "Frontend Developer I", "role_category": "frontend_developer",
             "skills_str": "javascript, html, css, react"},
            {"job_title": "Frontend Developer II", "role_category": "frontend_developer",
             "skills_str": "javascript, html, css"},
        ])

    if "skills_str" not in df.columns:
        raise ValueError("Expected a 'skills_str' column in the dataset")
    if "role_category" not in df.columns:
        df["role_category"] = "general"
    df["role_category"] = df["role_category"].fillna("general")

    df["skills"] = df["skills_str"].apply(parse_skill_list)
    return df


jobs_df = load_job_dataset(DATA_PATH)
print(f"{len(jobs_df)} postings across {jobs_df['role_category'].nunique()} role categories")
jobs_df.head()

'jobs_dataset_skills_final.csv' not found - using built-in sample dataset
11 postings across 4 role categories


## 4. Domain Model
Skill and JobRole classes used by the rule engine.

In [5]:
@dataclass
class JobRole:
    role_id: str
    title: str
    required_skills: List[str]
    preferred_skills: List[str] = field(default_factory=list)
    recommendation_text: str = ""
    postings_analyzed: int = 0

    def match_against(self, user_skills: set) -> Dict[str, Any]:
        req = set(self.required_skills)
        pref = set(self.preferred_skills)

        req_missing = sorted(req - user_skills)
        pref_missing = sorted(pref - user_skills)
        req_matched = len(req) - len(req_missing)
        pref_matched = len(pref) - len(pref_missing)

        # required skills weighted double, preferred skills weighted single
        total_weight = (len(req) * 2) + len(pref)
        earned_weight = (req_matched * 2) + pref_matched
        match_score = round(earned_weight / total_weight, 3) if total_weight else 0.0

        return {
            "role": self.title,
            "match_score": match_score,
            "required_matched": req_matched,
            "required_total": len(req),
            "preferred_matched": pref_matched,
            "preferred_total": len(pref),
            "required_missing": req_missing,
            "preferred_missing": pref_missing,
            "recommendation": self.recommendation_text,
        }

print("JobRole class defined")

JobRole class defined


## 5. Build Rule-Based Decision Engine
Derive IF-THEN rules from the job dataset: a skill is REQUIRED if it appears in most postings for a role, PREFERRED if it appears less often.

In [6]:
class RuleEngine:
    def __init__(self, jobs_df: pd.DataFrame,
                 required_threshold: float = 0.5,
                 preferred_threshold: float = 0.2):
        self.required_threshold = required_threshold
        self.preferred_threshold = preferred_threshold
        self.roles: List[JobRole] = self._build_rules(jobs_df)

    def _build_rules(self, jobs_df: pd.DataFrame) -> List[JobRole]:
        roles = []
        for category, group in jobs_df.groupby("role_category"):
            n_postings = len(group)
            skill_counts: Dict[str, int] = {}
            for skills in group["skills"]:
                for s in skills:
                    skill_counts[s] = skill_counts.get(s, 0) + 1

            required, preferred = [], []
            for skill, count in skill_counts.items():
                freq = count / n_postings
                if freq >= self.required_threshold:
                    required.append(skill)
                elif freq >= self.preferred_threshold:
                    preferred.append(skill)

            if not required and not preferred:
                continue

            title = category.replace("_", " ").title()
            roles.append(JobRole(
                role_id=category,
                title=title,
                required_skills=sorted(required),
                preferred_skills=sorted(preferred),
                postings_analyzed=n_postings,
                recommendation_text=(
                    f"Derived from {n_postings} '{title}' postings \u2014 "
                    f"required: {', '.join(sorted(required)) or 'none identified'}; "
                    f"preferred: {', '.join(sorted(preferred)) or 'none identified'}."
                ),
            ))
        return roles

    def evaluate(self, user_skills: List[str]) -> Dict[str, Any]:
        user_set = {normalize_text(s) for s in user_skills}
        results = [role.match_against(user_set) for role in self.roles]
        results.sort(key=lambda r: (-r["match_score"], len(r["required_missing"])))
        return {"rules_evaluated": len(results), "rules": results}


rule_engine = RuleEngine(jobs_df)
print(f"Rule engine built {len(rule_engine.roles)} role rules from the dataset:")
for role in rule_engine.roles:
    print(f"  - {role.title}: required={role.required_skills} | preferred={role.preferred_skills}")

Rule engine built 4 role rules from the dataset:
  - Backend Developer: required=['aws', 'docker', 'python', 'sql'] | preferred=['kubernetes']
  - Data Analyst: required=['excel', 'python', 'sql', 'statistics'] | preferred=['power bi', 'tableau']
  - Frontend Developer: required=['css', 'html', 'javascript', 'react'] | preferred=[]
  - Ml Engineer: required=['machine learning', 'python', 'scikit-learn', 'tensorflow'] | preferred=['deep learning', 'statistics']


In [7]:
user_profile = {
    "name": "Alice",
    "skills": ["Python", "SQL", "Data Visualization", "Communication"],
}

rule_output = rule_engine.evaluate(user_profile["skills"])
print(json.dumps(rule_output, indent=2))

{
  "rules_evaluated": 4,
  "rules": [
    {
      "role": "Backend Developer",
      "match_score": 0.444,
      "required_matched": 2,
      "required_total": 4,
      "preferred_matched": 0,
      "preferred_total": 1,
      "required_missing": [
        "aws",
        "docker"
      ],
      "preferred_missing": [
        "kubernetes"
      ],
      "recommendation": "Derived from 3 'Backend Developer' postings \u2014 required: aws, docker, python, sql; preferred: kubernetes."
    },
    {
      "role": "Data Analyst",
      "match_score": 0.4,
      "required_matched": 2,
      "required_total": 4,
      "preferred_matched": 0,
      "preferred_total": 2,
      "required_missing": [
        "excel",
        "statistics"
      ],
      "preferred_missing": [
        "power bi",
        "tableau"
      ],
      "recommendation": "Derived from 3 'Data Analyst' postings \u2014 required: excel, python, sql, statistics; preferred: power bi, tableau."
    },
    {
      "role": "Ml Engin

## 6. Generate DAG-Based Learning Path
Model skill dependencies as a directed acyclic graph and use topological sort to produce an ordered learning path for missing skills.

In [8]:
class LearningPathGenerator:
    DEFAULT_DEPENDENCIES: Dict[str, List[str]] = {
        "data structures": ["python basics"],
        "algorithms": ["data structures"],
        "sql": ["data structures"],
        "web frameworks": ["sql"],
        "statistics": ["python basics"],
        "excel": ["python basics"],
        "tensorflow": ["python basics"],
        "scikit-learn": ["python basics"],
        "machine learning": ["python basics", "statistics"],
        "deep learning": ["machine learning", "tensorflow"],
        "data visualization": ["python basics", "excel"],
        "docker": ["python basics"],
        "aws": ["docker"],
        "kubernetes": ["docker"],
        "power bi": ["excel"],
        "tableau": ["excel"],
        "react": ["javascript"],
    }

    def __init__(self, dependencies: Optional[Dict[str, List[str]]] = None):
        self.dependencies = dependencies or self.DEFAULT_DEPENDENCIES
        self.graph = self._build_graph(self.dependencies)

    @staticmethod
    def _build_graph(dependencies: Dict[str, List[str]]) -> nx.DiGraph:
        graph = nx.DiGraph()
        for skill, prereqs in dependencies.items():
            graph.add_node(skill)
            for prereq in prereqs:
                graph.add_edge(prereq, skill)
        if not nx.is_directed_acyclic_graph(graph):
            raise ValueError("Skill dependency graph must be acyclic.")
        return graph

    def generate_path(self, user_skills: List[str],
                       target_skills: Optional[List[str]] = None) -> List[str]:
        known = {normalize_text(s) for s in user_skills}

        if target_skills is not None:
            missing = [normalize_text(s) for s in target_skills if normalize_text(s) not in known]
        else:
            missing = [n for n in self.graph.nodes if n not in known]

        in_graph = [m for m in missing if m in self.graph.nodes]
        not_in_graph = [m for m in missing if m not in self.graph.nodes]
        if not_in_graph:
            print(f"No prerequisite mapping for: {not_in_graph}")

        if not in_graph:
            return []

        required_prereqs = set()
        for skill in in_graph:
            required_prereqs.update(
                p for p in nx.ancestors(self.graph, skill) if normalize_text(p) not in known
            )

        subgraph_nodes = required_prereqs.union(in_graph)
        subgraph = self.graph.subgraph(subgraph_nodes).copy()
        ordered = list(nx.topological_sort(subgraph))
        return [skill for skill in ordered if normalize_text(skill) not in known]


path_generator = LearningPathGenerator()
print(f"DAG built with {path_generator.graph.number_of_nodes()} skill nodes and "
      f"{path_generator.graph.number_of_edges()} prerequisite edges.")

DAG built with 19 skill nodes and 20 prerequisite edges.


In [9]:
learning_path: List[str] = []
if rule_output["rules"]:
    top_role = rule_output["rules"][0]
    missing_for_top_role = top_role["required_missing"] + top_role["preferred_missing"]
    learning_path = path_generator.generate_path(user_profile["skills"], missing_for_top_role)
    role_name = top_role["role"]
    role_score = top_role["match_score"]
    print(f"Top matched role: '{role_name}' (score={role_score})")
    if learning_path:
        print(f"Ordered learning path: {learning_path}")
    else:
        print("Ordered learning path: No dependency-mapped skills missing for this role.")
else:
    print("[RULE ENGINE] No roles could be evaluated - check the dataset.")

Top matched role: 'Backend Developer' (score=0.444)
Ordered learning path: ['python basics', 'docker', 'aws', 'kubernetes']


## 7. Persist Jobs and Rules to MongoDB
Disabled by default (USE_MONGODB=False). Set the flag to True and install pymongo on the database machine to enable this.

In [10]:
class DatabaseManager:
    def __init__(self, uri: str = MONGO_URI, db_name: str = MONGO_DB_NAME):
        from pymongo import MongoClient
        self.client = MongoClient(uri, serverSelectionTimeoutMS=3000)
        self.db = self.client[db_name]
        self.jobs_collection = self.db["jobs"]
        self.users_collection = self.db["users"]
        self.rules_collection = self.db["rules"]

    def save_jobs(self, jobs_df: pd.DataFrame) -> None:
        self.jobs_collection.delete_many({})
        self.jobs_collection.insert_many(jobs_df.to_dict(orient="records"))

    def save_rules(self, rule_engine: "RuleEngine") -> None:
        self.rules_collection.delete_many({})
        docs = [role.__dict__ for role in rule_engine.roles]
        self.rules_collection.insert_many(docs)

    def save_user_result(self, user_profile: Dict[str, Any],
                          rule_output: Dict[str, Any],
                          learning_path: List[str]) -> None:
        doc = {
            "name": user_profile.get("name"),
            "skills": user_profile.get("skills"),
            "rule_recommendations": rule_output,
            "learning_path": learning_path,
        }
        self.users_collection.delete_many({"name": doc["name"]})
        self.users_collection.insert_one(doc)

    def fetch_rules(self) -> List[Dict[str, Any]]:
        return list(self.rules_collection.find({}, {"_id": 0}))


if USE_MONGODB:
    db_manager = DatabaseManager()
    db_manager.save_jobs(jobs_df)
    db_manager.save_rules(rule_engine)
    db_manager.save_user_result(user_profile, rule_output, learning_path)
    print("[MongoDB] Persistence complete.")
else:
    print("[MongoDB] Skipped (USE_MONGODB=False). Flip the flag in the config "
          "cell on the database machine to enable persistence.")

[MongoDB] Skipped (USE_MONGODB=False). Flip the flag in the config cell on the database machine to enable persistence.


## 8. Expose Flask Endpoints for Recommendations
Disabled by default (RUN_FLASK_APP=False). Set the flag to True and install flask on the deployment machine to enable this.

In [11]:
def create_app(rule_engine: "RuleEngine", path_generator: "LearningPathGenerator"):
    from flask import Flask, request, jsonify

    app = Flask(__name__)

    @app.route("/recommend", methods=["POST"])
    def recommend():
        payload = request.json or {}
        user_skills = payload.get("skills", [])

        rule_output = rule_engine.evaluate(user_skills)
        missing_for_top_role = []
        if rule_output["rules"]:
            top_role = rule_output["rules"][0]
            missing_for_top_role = top_role["required_missing"] + top_role["preferred_missing"]

        path = path_generator.generate_path(user_skills, missing_for_top_role)

        return jsonify({
            "rule_recommendations": rule_output,
            "learning_path": path,
            "input_skills": user_skills,
        })

    @app.route("/rules", methods=["GET"])
    def list_rules():
        return jsonify({"rules": [role.__dict__ for role in rule_engine.roles]})

    return app


if RUN_FLASK_APP:
    app = create_app(rule_engine, path_generator)
    print("[Flask] Starting server on http://0.0.0.0:5000 ...")
    app.run(host="0.0.0.0", port=5000, debug=True)
else:
    print("[Flask] Skipped (RUN_FLASK_APP=False). Flip the flag in the config "
          "cell on the deployment machine to start the API server.")

[Flask] Skipped (RUN_FLASK_APP=False). Flip the flag in the config cell on the deployment machine to start the API server.


## 9. Demonstrate End-to-End Pipeline
Run a sample user through the rule engine and DAG learning path to show the complete workflow.

In [12]:
summary = {
    "user": user_profile,
    "top_recommendations": rule_output["rules"][:3],
    "learning_path_for_top_match": learning_path,
}
print(json.dumps(summary, indent=2))

{
  "user": {
    "name": "Alice",
    "skills": [
      "Python",
      "SQL",
      "Data Visualization",
      "Communication"
    ]
  },
  "top_recommendations": [
    {
      "role": "Backend Developer",
      "match_score": 0.444,
      "required_matched": 2,
      "required_total": 4,
      "preferred_matched": 0,
      "preferred_total": 1,
      "required_missing": [
        "aws",
        "docker"
      ],
      "preferred_missing": [
        "kubernetes"
      ],
      "recommendation": "Derived from 3 'Backend Developer' postings \u2014 required: aws, docker, python, sql; preferred: kubernetes."
    },
    {
      "role": "Data Analyst",
      "match_score": 0.4,
      "required_matched": 2,
      "required_total": 4,
      "preferred_matched": 0,
      "preferred_total": 2,
      "required_missing": [
        "excel",
        "statistics"
      ],
      "preferred_missing": [
        "power bi",
        "tableau"
      ],
      "recommendation": "Derived from 3 'Data Analy